# Modifying DataFrame columns

- Previously, you filtered out any rows that didn't conform to something generally resembling a name. Now based on your earlier work, your manager has asked you to create two new columns - `first_name` and `last_name`. She asks you to split the `VOTER_NAME` column into words on any space character. You'll treat the last word as the `last_name`, and all other words as the `first_name`. You'll be using some new functions in this exercise including `.split()`, `.size()`, and `.getItem()`. The `.getItem(index)` takes an integer value to return the appropriately numbered item in the column. The functions `.split()` and `.size()` are in the `pyspark.sql.functions` library.

- Please note that these operations are always somewhat specific to the use case. Having your data conform to a format often matters more than the specific details of the format. Rarely is a data cleaning task meant just for one person - matching a defined format allows for easier sharing of the data later (ie, Paul doesn't need to worry about names - Mary already cleaned the dataset).

- The filtered voter DataFrame from your previous exercise is available as `voter_df`. The `pyspark.sql.functions` library is available under the alias `F`.

## Instructions

- Add a new column called `splits` holding the list of possible names.
- Use the `getItem()` method and create a new column called `first_name`.
- Get the last entry of the `splits` list and create a column called `last_name`.
- Drop the `splits` column and show the new `voter_df`.

In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('yarn'). \
#     getOrCreate()

In [5]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [6]:
sc = spark.sparkContext

In [8]:
import pyspark.sql.functions as F

# Load the CSV file
voter_df = spark.read.format('csv').options(Header=True).load('file:///home/talentum/UBUNTU_HOME/test-jupyter/P3/M1/SM1/1_DataFramecolumnoperations/Dataset//DallasCouncilVoters.csv.gz')

# Add a new column called splits separated on whitespace
voter_df = voter_df.withColumn('splits', F.split(voter_df.VOTER_NAME, '\s+')) # here '\s+' means one or more whitespaces

print(voter_df.printSchema())

# Create a new column called first_name based on the first item in splits
voter_df = voter_df.withColumn('first_name', voter_df.splits.getItem(0))

# Get the last entry of the splits list and create a column called last_name
voter_df = voter_df.withColumn('last_column', voter_df.splits.getItem(F.size('splits') - 1))

# Drop the splits column
voter_df = voter_df.drop('splits')

# Show the voter_df DataFrame
voter_df.show()

root
 |-- DATE: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER_NAME: string (nullable = true)
 |-- splits: array (nullable = true)
 |    |-- element: string (containsNull = true)

None
+----------+-------------+-------------------+----------+-----------+
|      DATE|        TITLE|         VOTER_NAME|first_name|last_column|
+----------+-------------+-------------------+----------+-----------+
|02/08/2017|Councilmember|  Jennifer S. Gates|  Jennifer|      Gates|
|02/08/2017|Councilmember| Philip T. Kingston|    Philip|   Kingston|
|02/08/2017|        Mayor|Michael S. Rawlings|   Michael|   Rawlings|
|02/08/2017|Councilmember|       Adam Medrano|      Adam|    Medrano|
|02/08/2017|Councilmember|       Casey Thomas|     Casey|     Thomas|
|02/08/2017|Councilmember|Carolyn King Arnold|   Carolyn|     Arnold|
|02/08/2017|Councilmember|       Scott Griggs|     Scott|     Griggs|
|02/08/2017|Councilmember|   B. Adam  McGough|        B.|    McGough|
|02/08/2017|Council